In [1]:
# Imports
import os
from pathlib import Path
markers = (".git", "Program")
current_dir = Path.cwd()
project_root = next((path for path in (current_dir, *current_dir.parents) if any((path / marker).exists() for marker in markers)), current_dir)
os.chdir(project_root)

import concurrent.futures
import datetime as dt
from functools import partial
from fundamentals import *
from helper_functions import modify_current_date, get_df, get_infix
import matplotlib.pyplot as plt
import multiprocessing
import numpy as np
import pandas as pd
from plot import *
from technicals import *
from tqdm import tqdm

# Start of the program
start = dt.datetime.now()

# Variables
all_stocks = False
period_short = 60
period_long = 252
RS = 90
factors = [0.1, 0.55, 0.35]
backtest = False

# Index
index_name = "^GSPC"
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite"}

# Get the infix
infix = get_infix("^GSPC", index_dict, all_stocks) 

# Modify the current date
current_date = modify_current_date(start, index_name)

In [2]:
# Index
indices = ["^HSI", "^GSPC", "^IXIC", "QLD", "TQQQ"]
index_dict = {"^HSI": "Hang Seng Index", "^GSPC": "S&P 500", "^IXIC": "NASDAQ Composite", "QLD": "ProShares Ultra QQQ", "TQQQ": "ProShares UltraPro QQQ"}

# Momentum and Factor ETFs
etfs_dict = {
    # iShares
    "MTUM": "iShares MSCI USA Momentum Factor ETF",
    "IMTM": "iShares MSCI Intl Momentum Factor ETF",
    "SMLF": "iShares U.S. Small‑Cap Equity Factor ETF",
    "IUSV": "iShares S&P U.S. Value ETF",
    "IJJ": "iShares S&P Mid-Cap 400 Value ETF",
    "IJS": "iShares S&P Small-Cap 600 Value ETF",
    "EFV": "iShares MSCI EAFE Value ETF",
    "EWG": "iShares MSCI Germany ETF",
    "EWQ": "iShares MSCI France ETF",
    "EWU": "iShares MSCI United Kingdom ETF",
    "EWL": "iShares MSCI Switzerland ETF",
    "EWP": "iShares MSCI Spain ETF",
    "EWI": "iShares MSCI Italy ETF",
    "EWD": "iShares MSCI Sweden ETF",
    "EWK": "iShares MSCI Belgium ETF",
    "EWO": "iShares MSCI Austria ETF",
    "EWN": "iShares MSCI Netherlands ETF",
    "EWJ": "iShares MSCI Japan ETF",
    "EWY": "iShares MSCI South Korea ETF",
    "EWT": "iShares MSCI Taiwan ETF",
    "EWH": "iShares MSCI Hong Kong ETF",
    "EWS": "iShares MSCI Singapore ETF",
    "EWM": "iShares MSCI Malaysia ETF",
    "EWC": "iShares MSCI Canada ETF",
    "EWZ": "iShares MSCI Brazil ETF",
    "EWW": "iShares MSCI Mexico ETF",
    
    # Invesco
    "SPMO": "Invesco S&P 500 Momentum ETF",
    "XMMO": "Invesco S&P MidCap Momentum ETF",
    "XSMO": "Invesco S&P SmallCap Momentum ETF",
    "PDP": "Invesco Dorsey Wright Momentum ETF",
    "IDMO": "Invesco S&P Intl Developed Momentum ETF",
    "RPV": "Invesco S&P 500 Pure Value ETF",
    "SPHD": "Invesco S&P 500 High Dividend Low Volatility ETF",
    
    # Vanguard
    "VFMO": "Vanguard U.S. Momentum Factor ETF",
    "VYM": "Vanguard High Dividend Yield ETF",
    "VOE": "Vanguard S&P Mid-Cap Value ETF",
    "VONV": "Vanguard Russell 1000 Value ETF",
    "VOOV": "Vanguard S&P 500 Value ETF",
    "MGV": "Vanguard Mega Cap Value ETF",
    "VBR": "Vanguard Small-Cap Value ETF",
    "VTV": "Vanguard Value ETF",
    
    # Avantis
    "AVLV": "Avantis U.S. Large Cap Value ETF",
    "AVUV": "Avantis U.S. Small Cap Value ETF",
    
    # Others
    "AFLG": "First Trust Active Factor Large Cap Intl ETF",
    "CGDV": "Capital Group Dividend Value ETF",
    "DWMF": "WisdomTree International Multifactor ETF",
    "GLOV": "Activebeta World Low Vol Plus Equity ETF",
    "JMOM": "JPMorgan U.S. Momentum Factor ETF",
    "SCHD": "Schwab U.S. Dividend Equity ETF",
    "3070.HK": "Ping An of China CSI HK Dividend ETF",
    "DXJ": "WisdomTree Japan Hedged Equity Fund"
}
etfs = list(etfs_dict.keys())

In [3]:
# ETF Performance Metrics Calculation
# This cell computes a comprehensive set of performance metrics
# for each ETF/index, including CAGR, volatility, Sharpe ratio,
# Sortino ratio, maximum drawdown (MDD), Calmar ratio, and
# drawdown recovery statistics — over both the max available
# period and the most recent 5-year window.

RISK_FREE_RATE = 0.03   # Annual risk-free rate assumption (3%)
TRADING_DAYS  = 252     # Approximate trading days per year

# Period lengths (in trading days) mapped to their column labels
CAGR_PERIODS = {
    "1 Year CAGR (%)": TRADING_DAYS * 1,
    "3 Year CAGR (%)": TRADING_DAYS * 3,
    "5 Year CAGR (%)": TRADING_DAYS * 5,
    "10 Year CAGR (%)": TRADING_DAYS * 10,
    "20 Year CAGR (%)": TRADING_DAYS * 20,
}
PERIOD_5Y = TRADING_DAYS * 5

# Compute annualised return (CAGR) given price series
def calc_cagr(start_px, end_px, n_days):
    """Return CAGR in percent over n_days trading days."""
    years = n_days / TRADING_DAYS
    return ((end_px / start_px) ** (1 / years) - 1) * 100

# Compute Sharpe ratio (annualised, excess return)
def calc_sharpe(daily_rets):
    return (daily_rets.mean() * TRADING_DAYS - RISK_FREE_RATE) / \
           (daily_rets.std() * np.sqrt(TRADING_DAYS))

# Compute Sortino ratio (downside deviation only)
def calc_sortino(daily_rets):
    downside = daily_rets[daily_rets < 0]
    return (daily_rets.mean() * TRADING_DAYS - RISK_FREE_RATE) / \
           (downside.std() * np.sqrt(TRADING_DAYS))

# Compute Calmar ratio
def calc_calmar(daily_rets, mdd_pct):
    """mdd_pct is already in percent (negative value)."""
    return (daily_rets.mean() * TRADING_DAYS - RISK_FREE_RATE) / abs(mdd_pct / 100)

# Compute volatility-adjusted momentum:
def calc_vol_adj_momentum(closes, daily_rets):
    """Return volatility-adjusted momentum score."""
    if len(daily_rets) < 252:
        return np.nan
    ret_252_21 = closes.iloc[-21] / closes.iloc[-252] - 1
    vol_60 = daily_rets.tail(60).std()
    if vol_60 == 0:
        return np.nan
    return ret_252_21 / vol_60

# Initialise the results DataFrame
etfs_data = pd.DataFrame(columns=[
    "Symbol", "Establishment Date",
    # --- Volatility-adjusted momentum ---
    "Vol-Adj Momentum",
    # --- Rolling CAGR ---
    "1 Year CAGR (%)", "3 Year CAGR (%)", "5 Year CAGR (%)",
    "10 Year CAGR (%)", "20 Year CAGR (%)",
    # --- Full-history metrics ---
    "Max Period CAGR (%)", "Max Period Annual Volatility (%)",
    "Max Period Sharpe", "Max Period Sortino",
    "Max Period MDD (%)", "Max Period Calmar",
    # --- 5-year window metrics ---
    "5 Year Annual Volatility (%)", "5 Year Sharpe", "5 Year Sortino",
    "5 Year MDD (%)", "5 Year Calmar",
    # --- Drawdown recovery (5-year window) ---
    "5 Year Peak Date", "5 Year Recovery Date",
    "5 Year Recovery Period (days)",
])

# Main loop — iterate over every index and ETF symbol
for etf in tqdm(indices + etfs):

    # Resolve human-readable name; fall back to ticker if not found
    etf_name = {**index_dict, **etfs_dict}.get(etf, etf)
    etfs_data.loc[etf_name, "Symbol"] = etf

    # Fetch price history (full available history, adjusted)
    df = get_df(etf, current_date, max_period=True, adj=True)
    etfs_data.loc[etf_name, "Establishment Date"] = df.index[0].date()

    # Build daily-return and cumulative-return series
    df["Percent Change"] = df["Close"].pct_change(fill_method=None).dropna()
    df["Cumulative Return"] = (1 + df["Percent Change"]).cumprod()
    df["Peak"] = df["Cumulative Return"].cummax()
    df["Drawdown"] = (df["Cumulative Return"] - df["Peak"]) / df["Peak"]
    daily_rets = df["Percent Change"]
    drawdowns = df["Drawdown"]

    # Convenience slice: most-recent 5-year window
    has_5y = len(df) >= PERIOD_5Y
    if has_5y:
        daily_rets_5y = daily_rets.iloc[-PERIOD_5Y:]
        drawdowns_5y  = drawdowns.iloc[-PERIOD_5Y:]

    # Convenience reference to close prices
    closes = df["Close"]

    # Volatility-adjusted momentum
    etfs_data.loc[etf_name, "Vol-Adj Momentum"] = calc_vol_adj_momentum(closes, daily_rets)

    # Rolling CAGR for standard look-back periods
    for label, n_days in CAGR_PERIODS.items():
        if len(df) >= n_days:
            etfs_data.loc[etf_name, label] = calc_cagr(
                closes.iloc[-n_days], closes.iloc[-1], n_days
            )

    # Full-history (max-period) metrics
    # 4a. CAGR over entire history
    etfs_data.loc[etf_name, "Max Period CAGR (%)"] = calc_cagr(
        closes.iloc[0], closes.iloc[-1], len(df)
    )

    # Annualised volatility
    etfs_data.loc[etf_name, "Max Period Annual Volatility (%)"] = \
        daily_rets.std() * np.sqrt(TRADING_DAYS) * 100

    # Risk-adjusted ratios
    etfs_data.loc[etf_name, "Max Period Sharpe"]  = calc_sharpe(daily_rets)
    etfs_data.loc[etf_name, "Max Period Sortino"] = calc_sortino(daily_rets)

    # Maximum drawdown (most negative trough-to-peak decline)
    mdd = drawdowns.min() * 100 # convert to percent
    etfs_data.loc[etf_name, "Max Period MDD (%)"] = mdd

    # Calmar ratio
    etfs_data.loc[etf_name, "Max Period Calmar"] = calc_calmar(daily_rets, mdd)

    # 5. 5-year window metrics (skipped when history < 5 years)
    if has_5y:
        # Annualised volatility
        etfs_data.loc[etf_name, "5 Year Annual Volatility (%)"] = \
            daily_rets_5y.std() * np.sqrt(TRADING_DAYS) * 100

        # Risk-adjusted ratios
        etfs_data.loc[etf_name, "5 Year Sharpe"] = calc_sharpe(daily_rets_5y)
        etfs_data.loc[etf_name, "5 Year Sortino"] = calc_sortino(daily_rets_5y)

        # Maximum drawdown within the 5-year slice
        mdd_5y = drawdowns_5y.min() * 100
        mdd_date_5y = drawdowns_5y.idxmin()
        etfs_data.loc[etf_name, "5 Year MDD (%)"] = mdd_5y

        # Calmar ratio
        etfs_data.loc[etf_name, "5 Year Calmar"] = calc_calmar(daily_rets_5y, mdd_5y)

        # Drawdown recovery analysis (5-year window)
        # Find the peak date immediately before the MDD trough
        peak_mask_5y = (
            (df.index <= mdd_date_5y) &
            (df["Cumulative Return"] == df["Peak"])
        )
        peak_date_5y = df.index[peak_mask_5y][-1]
        etfs_data.loc[etf_name, "5 Year Peak Date"] = peak_date_5y.date()

        # Find the first date after the trough where price recovers to (or exceeds) the prior peak level
        peak_level = df.loc[peak_date_5y, "Cumulative Return"]
        rec_mask_5y = (
            (df.index > mdd_date_5y) &
            (df["Cumulative Return"] >= peak_level)
        )
        rec_dates = df.index[rec_mask_5y]

        if len(rec_dates) > 0:
            rec_date_5y = rec_dates[0]
            rec_period = (rec_date_5y - peak_date_5y).days
            etfs_data.loc[etf_name, "5 Year Recovery Date"]  = rec_date_5y.date()
            etfs_data.loc[etf_name, "5 Year Recovery Period (days)"] = rec_period

100%|██████████| 56/56 [00:01<00:00, 41.01it/s]


In [4]:
etfs_data

,Symbol,Establishment Date,Vol-Adj Momentum,1 Year CAGR (%),3 Year CAGR (%),5 Year CAGR (%),10 Year CAGR (%),20 Year CAGR (%),Max Period CAGR (%),Max Period Annual Volatility (%),...,Max Period MDD (%),Max Period Calmar,5 Year Annual Volatility (%),5 Year Sharpe,5 Year Sortino,5 Year MDD (%),5 Year Calmar,5 Year Peak Date,5 Year Recovery Date,5 Year Recovery Period (days)
Hang Seng Index,^HSI,1986-12-31,19.35769,21.082345,6.851831,-0.910848,1.541845,2.793995,6.26919,25.588934,...,-65.18186,0.098327,25.130533,-0.024794,-0.037086,-55.700772,-0.011186,2018-01-26,NaN,NaN
S&P 500,^GSPC,1927-12-30,18.653,12.946255,18.545058,11.931858,13.993992,8.85708,6.29233,18.931636,...,-86.189579,0.056838,16.88114,0.573739,0.788265,-25.425097,0.380937,2022-01-03,2024-01-19,746
NASDAQ Composite,^IXIC,1971-02-05,18.30015,14.646119,23.769673,10.254021,18.095426,12.255125,10.369415,20.132115,...,-77.932386,0.114179,22.586916,0.40909,0.575218,-36.39528,0.253882,2021-11-19,2024-02-29,832
ProShares Ultra QQQ,QLD,2006-06-21,11.168773,16.103052,43.70741,16.153503,34.016528,NaN,24.115216,43.933621,...,-83.128873,0.340561,45.129206,0.488313,0.688338,-63.684475,0.346037,2021-11-19,2024-05-24,917
ProShares UltraPro QQQ,TQQQ,2010-02-11,7.545487,13.046506,58.674174,13.89842,42.21494,NaN,40.956896,60.996288,...,-81.659846,0.614271,67.075698,0.481973,0.677944,-81.659846,0.395894,2021-11-19,2024-12-04,1111
iShares MSCI USA Momentum Factor ETF,MTUM,2013-04-18,11.837239,14.254138,22.23986,8.699506,15.787471,NaN,14.74497,19.516867,...,-34.082278,0.371794,20.669114,0.359298,0.497084,-32.284553,0.230029,2021-11-03,2024-03-04,852
iShares MSCI Intl Momentum Factor ETF,IMTM,2015-01-27,33.217298,35.88871,21.54941,9.42573,11.761066,NaN,9.348588,19.930708,...,-30.683154,0.257707,17.019001,0.444027,0.655393,-30.683154,0.246288,2021-11-08,2024-02-22,836
iShares U.S. Small‑Cap Equity Factor ETF,SMLF,2015-04-30,17.13137,16.008817,14.426886,10.313036,13.393771,NaN,11.096369,21.637076,...,-41.89219,0.235898,21.253902,0.421397,0.653679,-26.277094,0.340842,2024-11-25,2025-08-28,276
iShares S&P U.S. Value ETF,IUSV,2000-08-04,18.9491,13.466734,14.078725,12.660964,13.201276,8.842437,8.443116,19.077951,...,-60.183538,0.115185,14.667803,0.684584,0.98442,-17.951545,0.559359,2022-04-20,2023-02-01,287
iShares S&P Mid-Cap 400 Value ETF,IJJ,2000-07-28,12.295056,13.815384,9.964862,10.43315,12.395682,9.012632,10.410317,21.846414,...,-58.003737,0.160365,19.81091,0.451777,0.695865,-22.67915,0.394641,2024-11-25,2025-12-10,380


In [5]:
# Save the data as a CSV file
result_folder = "Result"
filename = os.path.join(result_folder, "etfs_comparison.csv")
etfs_data.to_csv(filename)